In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

GPU available: True
Device name: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
import os
from pathlib import Path

# Full local SubPipe dataset (5 survey chunks, ~101GB) — not the "Mini" Zenodo
# subset the original version of this notebook downloaded. Already on disk.
SUBPIPE_ROOT = Path("/home/nial-rojan/SIH 2026/Datasets/Sub_pipe/SubPipe")
CHUNKS = ["Chunk0", "Chunk1", "Chunk2", "Chunk3", "Chunk4"]

assert SUBPIPE_ROOT.exists(), f"SubPipe dataset not found at {SUBPIPE_ROOT}"
print("Using local SubPipe dataset at:", SUBPIPE_ROOT)

Using local SubPipe dataset at: /home/nial-rojan/SIH 2026/Datasets/Sub_pipe/SubPipe


In [3]:
# Sanity check: confirm each chunk's HF sonar images are present before processing
for chunk in CHUNKS:
    img_dir = SUBPIPE_ROOT / "DATA" / chunk / "SSS_HF_images" / "Image"
    n_images = len(list(img_dir.glob("*.pbm"))) if img_dir.exists() else 0
    print(f"{chunk}: {n_images} HF images")

Chunk0: 903 HF images
Chunk1: 541 HF images
Chunk2: 539 HF images
Chunk3: 190 HF images
Chunk4: 2749 HF images


In [4]:
for chunk in CHUNKS:
    chunk_root = SUBPIPE_ROOT / "DATA" / chunk
    print(f"{chunk}/")
    for sub in ["SSS_HF_images", "SSS_LF_images"]:
        sub_dir = chunk_root / sub
        if not sub_dir.exists():
            continue
        for name in ["Image", "YOLO_Annotation", "COCO_Annotation"]:
            d = sub_dir / name
            n = len(list(d.glob("*"))) if d.exists() else 0
            print(f"  {sub}/{name}/  ({n} files)")

Chunk0/
  SSS_HF_images/Image/  (1011 files)
  SSS_HF_images/YOLO_Annotation/  (671 files)
  SSS_HF_images/COCO_Annotation/  (1 files)
  SSS_LF_images/Image/  (1055 files)
  SSS_LF_images/YOLO_Annotation/  (697 files)
  SSS_LF_images/COCO_Annotation/  (1 files)
Chunk1/
  SSS_HF_images/Image/  (541 files)
  SSS_HF_images/YOLO_Annotation/  (248 files)
  SSS_HF_images/COCO_Annotation/  (1 files)
  SSS_LF_images/Image/  (541 files)
  SSS_LF_images/YOLO_Annotation/  (244 files)
  SSS_LF_images/COCO_Annotation/  (1 files)
Chunk2/
  SSS_HF_images/Image/  (539 files)
  SSS_HF_images/YOLO_Annotation/  (436 files)
  SSS_HF_images/COCO_Annotation/  (1 files)
  SSS_LF_images/Image/  (539 files)
  SSS_LF_images/YOLO_Annotation/  (456 files)
  SSS_LF_images/COCO_Annotation/  (1 files)
Chunk3/
  SSS_HF_images/Image/  (190 files)
  SSS_HF_images/YOLO_Annotation/  (30 files)
  SSS_HF_images/COCO_Annotation/  (1 files)
  SSS_LF_images/Image/  (189 files)
  SSS_LF_images/YOLO_Annotation/  (32 files)
  SS

In [5]:
import json
from collections import Counter

for freq in ["HF", "LF"]:
    total = Counter()
    for chunk in CHUNKS:
        coco_path = SUBPIPE_ROOT / "DATA" / chunk / f"SSS_{freq}_images" / "COCO_Annotation" / "coco_format.json"
        if not coco_path.exists():
            continue
        with open(coco_path) as f:
            coco = json.load(f)
        cats = {c["id"]: c["name"] for c in coco["categories"]}
        counts = Counter(cats[ann["category_id"]] for ann in coco["annotations"])
        print(f"  {chunk} {freq}: {dict(counts)}")
        total.update(counts)
    print(f"--- {freq} total across all chunks: {dict(total)} ---\n")

  Chunk0 HF: {'Pipeline': 593}
  Chunk1 HF: {'Pipeline': 247}
  Chunk2 HF: {'Pipeline': 435}
  Chunk3 HF: {'Pipeline': 29}
  Chunk4 HF: {'Pipeline': 1818}
--- HF total across all chunks: {'Pipeline': 3122} ---

  Chunk0 LF: {'Pipeline': 726}
  Chunk1 LF: {'Pipeline': 243}
  Chunk2 LF: {'Pipeline': 455}
  Chunk3 LF: {'Pipeline': 31}
  Chunk4 LF: {'Pipeline': 1668}
--- LF total across all chunks: {'Pipeline': 3123} ---



In [ ]:
import os, shutil
from pathlib import Path
from PIL import Image

freq = "HF"
out_root = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_ready"))

# Pool every chunk's HF frames. The 5 chunks are one continuous survey mission
# recorded back-to-back (Chunk0 ends exactly where Chunk1 begins, etc.), so
# sorting the pooled set by timestamp reproduces the true chronological order.
# Kept un-shuffled for the same reason as before: a random shuffle would leak
# near-duplicate consecutive sonar pings between train and val.
samples = []
for chunk in CHUNKS:
    img_dir = SUBPIPE_ROOT / "DATA" / chunk / f"SSS_{freq}_images" / "Image"
    label_dir = SUBPIPE_ROOT / "DATA" / chunk / f"SSS_{freq}_images" / "YOLO_Annotation"
    for img_path in img_dir.glob("*.pbm"):
        samples.append((float(img_path.stem), img_path, label_dir / f"{img_path.stem}.txt"))

samples.sort(key=lambda s: s[0])
split_idx = int(len(samples) * 0.8)
splits = {"train": samples[:split_idx], "val": samples[split_idx:]}

counts = {"train": 0, "val": 0}
for split, items in splits.items():
    for _, img_path, src_label in items:
        stem = img_path.stem
        dst_img = out_root / split / "images" / f"{stem}.png"
        dst_img.parent.mkdir(parents=True, exist_ok=True)
        Image.open(img_path).convert("L").save(dst_img)

        dst_label = out_root / split / "labels" / f"{stem}.txt"
        dst_label.parent.mkdir(parents=True, exist_ok=True)
        if src_label.exists():
            shutil.copy(src_label, dst_label)
        else:
            dst_label.touch()  # no pipe visible in this frame — valid negative example
        counts[split] += 1

yaml_content = f"""path: {out_root.resolve()}
train: train/images
val: val/images
names:
  0: Pipeline
"""
(out_root / "data.yaml").write_text(yaml_content)
print(counts)
print("wrote", out_root / "data.yaml")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano — fastest, good fit for a 6GB laptop GPU

model.train(
    data=str(out_root / "data.yaml"),
    epochs=30,           # short sanity run first
    imgsz=640,
    batch=8,              # conservative for 6GB VRAM — bump to 16 later if it fits
    dropout=0.1,          # keep >0 — needed later for MC Dropout confidence scoring
    hsv_h=0.0, hsv_s=0.0, # sonar is grayscale — color augmentation is meaningless here
    hsv_v=0.2,
    mosaic=0.3,
    mixup=0.0,
    degrees=10.0,
    translate=0.1,
    scale=0.3,
    shear=2.0,
    patience=15,
)

In [ ]:
results = model.predict(
    source=str(out_root / "val" / "images"),
    save=True,
    conf=0.25
)
print("saved predictions to:", results[0].save_dir)

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

pred_dir = str(results[0].save_dir)
pred_images = sorted(os.listdir(pred_dir))[:6]  # first 6

fig, axes = plt.subplots(2, 3, figsize=(18, 6))
for ax, fname in zip(axes.flat, pred_images):
    img = Image.open(os.path.join(pred_dir, fname))
    ax.imshow(img, cmap='gray')
    ax.set_title(fname, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import os
from pathlib import Path
from PIL import Image

def load_yolo_labels(label_path):
    """Returns list of (cls, xc, yc, w, h) all normalized 0-1, relative to full image."""
    if not label_path.exists():
        return []
    boxes = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
                boxes.append((cls, xc, yc, w, h))
    return boxes

def tile_image_and_labels(img_path, label_path, out_img_dir, out_label_dir, tile_size=500, overlap=100):
    img = Image.open(img_path).convert("L")
    W, H = img.size
    boxes = load_yolo_labels(label_path)

    stride = tile_size - overlap
    x_starts = list(range(0, max(W - tile_size, 0) + 1, stride))
    if x_starts[-1] + tile_size < W:
        x_starts.append(W - tile_size)  # make sure the right edge is covered

    stem = img_path.stem
    for i, x0 in enumerate(x_starts):
        x1 = x0 + tile_size
        tile = img.crop((x0, 0, x1, H))

        tile_boxes = []
        for cls, xc, yc, w, h in boxes:
            abs_xc, abs_yc, abs_w, abs_h = xc * W, yc * H, w * W, h * H
            box_x0, box_x1 = abs_xc - abs_w / 2, abs_xc + abs_w / 2

            inter_x0, inter_x1 = max(box_x0, x0), min(box_x1, x1)
            if inter_x1 <= inter_x0:
                continue  # box doesn't overlap this tile at all

            overlap_frac = (inter_x1 - inter_x0) / abs_w
            if overlap_frac < 0.3:
                continue  # too sliver-thin a fragment to be a useful training example

            # clip box to tile bounds, convert back to tile-relative normalized coords
            new_xc = ((inter_x0 + inter_x1) / 2 - x0) / tile_size
            new_w = (inter_x1 - inter_x0) / tile_size
            tile_boxes.append((cls, new_xc, yc, new_w, h * H / tile_size))

        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_label_dir.mkdir(parents=True, exist_ok=True)
        tile.save(out_img_dir / f"{stem}_tile{i}.png")
        with open(out_label_dir / f"{stem}_tile{i}.txt", "w") as f:
            for cls, xc, yc, w, h in tile_boxes:
                f.write(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

print("function defined")

In [ ]:
def tile_image_and_labels_rect(
    img_path,
    label_path,
    out_img_dir,
    out_label_dir,
    tile_width=1000,
    tile_height=500,
    overlap=200
):
    img = Image.open(img_path).convert("L")
    W, H = img.size
    boxes = load_yolo_labels(label_path)

    # Horizontal sliding window
    stride = tile_width - overlap

    x_starts = list(
        range(
            0,
            max(W - tile_width, 0) + 1,
            stride
        )
    )

    # Make sure the right edge is always covered
    if not x_starts:
        x_starts = [0]

    if x_starts[-1] + tile_width < W:
        x_starts.append(W - tile_width)

    stem = img_path.stem

    for i, x0 in enumerate(x_starts):

        x1 = min(x0 + tile_width, W)

        # Full image height (500 px)
        tile = img.crop((x0, 0, x1, H))

        actual_tile_width = x1 - x0

        tile_boxes = []

        for cls, xc, yc, w, h in boxes:

            # Original normalized YOLO → absolute pixels
            abs_xc = xc * W
            abs_yc = yc * H
            abs_w = w * W
            abs_h = h * H

            # Original bounding-box boundaries
            box_x0 = abs_xc - abs_w / 2
            box_x1 = abs_xc + abs_w / 2

            # Intersection with tile
            inter_x0 = max(box_x0, x0)
            inter_x1 = min(box_x1, x1)

            if inter_x1 <= inter_x0:
                continue

            # How much of the original object remains?
            overlap_frac = (inter_x1 - inter_x0) / abs_w

            if overlap_frac < 0.3:
                continue

            # Convert clipped box to tile-relative coordinates
            new_xc = (
                ((inter_x0 + inter_x1) / 2 - x0)
                / actual_tile_width
            )

            new_w = (
                (inter_x1 - inter_x0)
                / actual_tile_width
            )

            # Vertical coordinates remain relative to full 500 px height
            new_yc = yc
            new_h = abs_h / H

            tile_boxes.append(
                (cls, new_xc, new_yc, new_w, new_h)
            )

        # Create output directories
        out_img_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        out_label_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        # Save tile
        tile.save(
            out_img_dir /
            f"{stem}_tile{i}.png"
        )

        # Save YOLO labels
        with open(
            out_label_dir /
            f"{stem}_tile{i}.txt",
            "w"
        ) as f:

            for cls, xc, yc, w, h in tile_boxes:
                f.write(
                    f"{cls} "
                    f"{xc:.6f} "
                    f"{yc:.6f} "
                    f"{w:.6f} "
                    f"{h:.6f}\n"
                )

print("✅ Rectangular tiling function defined")

In [ ]:
from pathlib import Path
import os

# Original dataset
DATASET = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_ready"))

# New tiled dataset
TILED = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled"))

# Make sure output directories exist
for split in ["train", "val"]:
    (TILED / split / "images").mkdir(parents=True, exist_ok=True)
    (TILED / split / "labels").mkdir(parents=True, exist_ok=True)

# Tile both train and validation sets
for split in ["train", "val"]:
    img_dir = DATASET / split / "images"
    label_dir = DATASET / split / "labels"

    out_img_dir = TILED / split / "images"
    out_label_dir = TILED / split / "labels"

    images = sorted(img_dir.glob("*.png"))

    print(f"\n{split.upper()}: {len(images)} images")

    for n, img_path in enumerate(images, 1):
        label_path = label_dir / f"{img_path.stem}.txt"

        tile_image_and_labels(
            img_path,
            label_path,
            out_img_dir,
            out_label_dir,
            tile_size=500,
            overlap=100
        )

        if n % 50 == 0 or n == len(images):
            print(f"  processed {n}/{len(images)}")

print("\n✅ TILING COMPLETE")
print(f"Output: {TILED}")

In [ ]:
from pathlib import Path
import os
from PIL import Image

TILED = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled"))

for split in ["train", "val"]:
    img_dir = TILED / split / "images"
    label_dir = TILED / split / "labels"

    images = sorted(img_dir.glob("*.png"))
    labels = sorted(label_dir.glob("*.txt"))

    print(f"\n{split.upper()}")
    print(f"Images : {len(images)}")
    print(f"Labels : {len(labels)}")

    # Check image dimensions
    sizes = {}
    for img_path in images[:100]:
        with Image.open(img_path) as im:
            sizes[im.size] = sizes.get(im.size, 0) + 1

    print("Image sizes (sample):", sizes)

    # Count labelled tiles and bounding boxes
    labelled_tiles = 0
    total_boxes = 0

    for label_path in labels:
        text = label_path.read_text().strip()

        if text:
            labelled_tiles += 1
            total_boxes += len(text.splitlines())

    print(f"Tiles containing objects: {labelled_tiles}")
    print(f"Total bounding boxes: {total_boxes}")

In [ ]:
from pathlib import Path
import os
from PIL import Image, ImageDraw
import random
import matplotlib.pyplot as plt

TILED = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled"))

# Pick random labelled training tiles
img_dir = TILED / "train" / "images"
label_dir = TILED / "train" / "labels"

labelled = []

for label_path in label_dir.glob("*.txt"):
    if label_path.read_text().strip():
        img_path = img_dir / f"{label_path.stem}.png"
        if img_path.exists():
            labelled.append((img_path, label_path))

print(f"Labelled tiles available: {len(labelled)}")

# Select up to 6 examples
samples = random.sample(labelled, min(6, len(labelled)))

fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 7)
)

axes = axes.flatten()

for ax, (img_path, label_path) in zip(axes, samples):

    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    W, H = img.size

    for line in label_path.read_text().strip().splitlines():

        cls, xc, yc, w, h = map(float, line.split())

        # YOLO normalized → pixel coordinates
        xc *= W
        yc *= H
        w *= W
        h *= H

        x0 = xc - w / 2
        y0 = yc - h / 2
        x1 = xc + w / 2
        y1 = yc + h / 2

        draw.rectangle(
            [x0, y0, x1, y1],
            outline="red",
            width=3
        )

        draw.text(
            (x0, max(0, y0 - 15)),
            f"class {int(cls)}",
            fill="red"
        )

    ax.imshow(img, cmap="gray")
    ax.set_title(img_path.name)
    ax.axis("off")

# Hide unused axes
for ax in axes[len(samples):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import os
import numpy as np

ORIGINAL = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_ready"))
TILED = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled"))

def label_stats(label_dir):
    widths = []
    heights = []
    boxes = 0

    for p in label_dir.glob("*.txt"):
        text = p.read_text().strip()

        if not text:
            continue

        for line in text.splitlines():
            parts = line.split()

            if len(parts) != 5:
                continue

            cls, xc, yc, w, h = map(float, parts)

            boxes += 1
            widths.append(w)
            heights.append(h)

    return boxes, widths, heights


for split in ["train", "val"]:
    print(f"\n{'='*50}")
    print(split.upper())

    # Original
    boxes, widths, heights = label_stats(
        ORIGINAL / split / "labels"
    )

    print("\nORIGINAL LABELS")
    print("Boxes:", boxes)

    if boxes:
        print(f"Width  mean: {np.mean(widths):.4f}")
        print(f"Width  min : {np.min(widths):.4f}")
        print(f"Width  max : {np.max(widths):.4f}")
        print(f"Height mean: {np.mean(heights):.4f}")
        print(f"Height min : {np.min(heights):.4f}")
        print(f"Height max : {np.max(heights):.4f}")

    # Tiled
    boxes, widths, heights = label_stats(
        TILED / split / "labels"
    )

    print("\nTILED LABELS")
    print("Boxes:", boxes)

    if boxes:
        print(f"Width  mean: {np.mean(widths):.4f}")
        print(f"Width  min : {np.min(widths):.4f}")
        print(f"Width  max : {np.max(widths):.4f}")
        print(f"Height mean: {np.mean(heights):.4f}")
        print(f"Height min : {np.min(heights):.4f}")
        print(f"Height max : {np.max(heights):.4f}")

In [ ]:
# NOTE: this square 1000x1000 tiling pass writes into the same yolo_tiled_1000
# folder that the next cell (rectangular 1000x500 tiling, the version actually
# used for training below) overwrites anyway — skipping it here to avoid a
# redundant pass over ~4,900 images. Left as a comment for reference:
#
# DATASET = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_ready"))
# TILED_1000 = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled_1000"))
# for split in ["train", "val"]:
#     ... tile_image_and_labels(..., tile_size=1000, overlap=200) ...
print("Skipped — superseded by the rectangular 1000x500 tiling cell below.")

In [ ]:
import inspect

# tile_image_and_labels_rect is the tiling function actually used for the
# training data below (rectangular 1000x500 tiles, preserving sonar height)
print(inspect.getsource(tile_image_and_labels_rect))

In [ ]:
from pathlib import Path
import os

DATASET = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_ready"))
TILED_1000 = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled_1000"))

for split in ["train", "val"]:

    img_dir = DATASET / split / "images"
    label_dir = DATASET / split / "labels"

    out_img_dir = TILED_1000 / split / "images"
    out_label_dir = TILED_1000 / split / "labels"

    images = sorted(img_dir.glob("*.png"))

    print(f"\n{split.upper()}: {len(images)} images")

    for n, img_path in enumerate(images, 1):

        label_path = label_dir / f"{img_path.stem}.txt"

        tile_image_and_labels_rect(
            img_path,
            label_path,
            out_img_dir,
            out_label_dir,
            tile_width=1000,
            tile_height=500,
            overlap=200
        )

        if n % 50 == 0 or n == len(images):
            print(f"  processed {n}/{len(images)}")

print("\n✅ 1000×500 TILING COMPLETE")
print(f"Output: {TILED_1000}")

In [ ]:
from pathlib import Path
import os
from PIL import Image
import numpy as np

TILED = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled_1000"))

def get_stats(label_dir):
    widths = []
    heights = []
    boxes = 0
    labelled_tiles = 0

    for p in label_dir.glob("*.txt"):
        text = p.read_text().strip()

        if not text:
            continue

        labelled_tiles += 1

        for line in text.splitlines():
            parts = line.split()

            if len(parts) != 5:
                continue

            cls, xc, yc, w, h = map(float, parts)

            boxes += 1
            widths.append(w)
            heights.append(h)

    return labelled_tiles, boxes, widths, heights


for split in ["train", "val"]:

    img_dir = TILED / split / "images"
    label_dir = TILED / split / "labels"

    images = sorted(img_dir.glob("*.png"))
    labels = sorted(label_dir.glob("*.txt"))

    print("\n" + "=" * 50)
    print(split.upper())

    print("Images:", len(images))
    print("Labels:", len(labels))

    # Check image dimensions
    sizes = {}

    for img_path in images[:100]:
        with Image.open(img_path) as im:
            sizes[im.size] = sizes.get(im.size, 0) + 1

    print("Image sizes:", sizes)

    labelled_tiles, boxes, widths, heights = get_stats(label_dir)

    print("Tiles containing objects:", labelled_tiles)
    print("Total bounding boxes:", boxes)

    if boxes:
        print(f"Box width  mean: {np.mean(widths):.4f}")
        print(f"Box width  min : {np.min(widths):.4f}")
        print(f"Box width  max : {np.max(widths):.4f}")
        print(f"Box height mean: {np.mean(heights):.4f}")
        print(f"Box height min : {np.min(heights):.4f}")
        print(f"Box height max : {np.max(heights):.4f}")

In [ ]:
import os
import random
from pathlib import Path

random.seed(0)

SRC = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled_1000"))
BALANCED = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled_1000_balanced"))

# ~89% of tiles were pure background, which was suppressing recall.
# Keep every positive tile, cap negatives at this ratio per positive.
NEG_TO_POS_RATIO = 2

def link_pair(img_src, lbl_src, img_dst_dir, lbl_dst_dir):
    img_dst_dir.mkdir(parents=True, exist_ok=True)
    lbl_dst_dir.mkdir(parents=True, exist_ok=True)
    img_dst = img_dst_dir / img_src.name
    lbl_dst = lbl_dst_dir / lbl_src.name
    if not img_dst.exists():
        os.symlink(img_src, img_dst)
    if not lbl_dst.exists():
        os.symlink(lbl_src, lbl_dst)

# Only rebalance TRAIN. Val stays untouched so metrics reflect the real distribution.
img_dir = SRC / "train" / "images"
label_dir = SRC / "train" / "labels"
out_img_dir = BALANCED / "train" / "images"
out_label_dir = BALANCED / "train" / "labels"

positives, negatives = [], []
for lbl in label_dir.glob("*.txt"):
    img = img_dir / f"{lbl.stem}.png"
    if not img.exists():
        continue
    if lbl.read_text().strip():
        positives.append((img, lbl))
    else:
        negatives.append((img, lbl))

keep_neg_n = min(len(negatives), len(positives) * NEG_TO_POS_RATIO)
kept_negatives = random.sample(negatives, keep_neg_n)

for img, lbl in positives + kept_negatives:
    link_pair(img, lbl, out_img_dir, out_label_dir)

print(f"positives: {len(positives)}")
print(f"negatives kept: {keep_neg_n} / {len(negatives)} available (ratio {NEG_TO_POS_RATIO}:1)")
print(f"new train set size: {len(positives) + keep_neg_n}  (was {len(positives) + len(negatives)})")
print(f"val split left untouched at: {SRC / 'val'}")

data_yaml = f"""path: {BALANCED}
train: train/images
val: ../yolo_tiled_1000/val/images

nc: 1
names:
  0: Pipeline
"""
(BALANCED / "data.yaml").write_text(data_yaml)
print("\nWrote data.yaml:\n" + data_yaml)


positives: 2635
negatives kept: 5270 / 20987 available (ratio 2:1)
new train set size: 7905  (was 23622)
val split left untouched at: /home/nial-rojan/sonar-debris/yolo_tiled_1000/val

Wrote data.yaml:
path: /home/nial-rojan/sonar-debris/yolo_tiled_1000_balanced
train: train/images
val: ../yolo_tiled_1000/val/images

nc: 1
names:
  0: Pipeline



In [ ]:
from pathlib import Path
import os
from PIL import Image, ImageDraw
import random
import matplotlib.pyplot as plt

TILED = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled_1000"))

img_dir = TILED / "train" / "images"
label_dir = TILED / "train" / "labels"

labelled = []

for label_path in label_dir.glob("*.txt"):
    if label_path.read_text().strip():
        img_path = img_dir / f"{label_path.stem}.png"

        if img_path.exists():
            labelled.append((img_path, label_path))

print("Labelled tiles:", len(labelled))

samples = random.sample(labelled, min(6, len(labelled)))

fig, axes = plt.subplots(
    2, 3,
    figsize=(18, 7)
)

axes = axes.flatten()

for ax, (img_path, label_path) in zip(axes, samples):

    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    W, H = img.size

    for line in label_path.read_text().strip().splitlines():

        cls, xc, yc, w, h = map(float, line.split())

        # YOLO normalized → pixels
        xc *= W
        yc *= H
        w *= W
        h *= H

        x0 = xc - w / 2
        y0 = yc - h / 2
        x1 = xc + w / 2
        y1 = yc + h / 2

        draw.rectangle(
            [x0, y0, x1, y1],
            outline="red",
            width=4
        )

        draw.text(
            (x0, max(0, y0 - 18)),
            f"class {int(cls)}",
            fill="red"
        )

    ax.imshow(img)
    ax.set_title(img_path.name)
    ax.axis("off")

for ax in axes[len(samples):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import os
import yaml

TILED = Path(os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled_1000"))

data = {
    "path": str(TILED),
    "train": "train/images",
    "val": "val/images",
    "nc": 1,
    "names": ["Pipeline"]
}

yaml_path = TILED / "data.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(yaml_path)
print("\n--- data.yaml ---")

print(yaml_path.read_text())

In [ ]:
!pip install ultralytics --break-system-packages


In [1]:
from ultralytics import YOLO
import os

RUNS_DIR = os.path.expanduser("~/SIH 2026/sonar-debris/runs/detect")

model = YOLO("yolov8n.pt")

results = model.train(
    data=os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled_1000/data.yaml"),

    # First proper training run
    epochs=50,
    imgsz=640,
    batch=8,

    # Preserve the sonar aspect ratio during training
    rect=True,

    # Conservative augmentation for sonar
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.2,
    mosaic=0.3,
    mixup=0.0,
    degrees=10.0,
    translate=0.1,
    scale=0.3,
    shear=2.0,

    # Training control
    patience=15,

    # Don't overwrite your previous experiment
    project=RUNS_DIR,
    name="pipeline_tiled_1000x500",
    exist_ok=True,

    # device=0 selects your first CUDA GPU (RTX 4060 8GB here — bump batch if VRAM allows)
    device=0
)

Ultralytics 8.4.135 🚀 Python-3.14.4 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7808MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/nial-rojan/sonar-debris/yolo_tiled_1000/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.3, multi_s

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import os

RUNS_DIR = os.path.expanduser("~/SIH 2026/sonar-debris/runs/detect")

# Load the best model from your training run
model = YOLO(
    f"{RUNS_DIR}/pipeline_tiled_1000x500/weights/best.pt"
)

# Validation tiles
val_images = os.path.expanduser("~/SIH 2026/sonar-debris/yolo_tiled_1000/val/images")

# Run inference
results = model.predict(
    source=val_images,
    save=True,
    conf=0.25,
    device=0,
    project=RUNS_DIR,
    name="pipeline_tiled_1000x500_predictions",
    exist_ok=True
)

print("✅ Predictions complete")
print("Saved to:", results[0].save_dir)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/5910 /home/nial-rojan/sonar-debris/yolo_tiled_1000/val/images/1693577628.310_tile0.png: 320x640 (no detections), 4.4ms
image 2/5910 /home/nial-rojan/sonar-debris/yolo_tiled_1000/val/images/1693577628.310_tile1.png: 320x640 1 Pipeline, 4.7ms
image 3/5910 /home/nial-rojan/sonar-debris/yolo_tiled_1000/val/images/1693577628.310_tile2.png: 320x640 (no detections), 6.9ms
image 4/5910 /home/nial-rojan/sonar-debris/yolo_tiled_1000/val/images/1693577628.31

In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt

# Keep only images where YOLO detected something
detected = [r for r in results if len(r.boxes) > 0]

print("Tiles with detections:", len(detected))

if not detected:
    print("No detections found.")
else:
    samples = random.sample(detected, min(6, len(detected)))

    fig, axes = plt.subplots(2, 3, figsize=(18, 7))
    axes = axes.flatten()

    for ax, r in zip(axes, samples):
        plotted = r.plot()

        # Ultralytics plot() returns BGR; matplotlib expects RGB
        ax.imshow(plotted[..., ::-1])
        ax.set_title(Path(r.path).name, fontsize=9)
        ax.axis("off")

    for ax in axes[len(samples):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Summary of confidence scores and number of detections

import numpy as np

all_conf = []
num_boxes = []

for r in results:
    if len(r.boxes) > 0:
        confs = r.boxes.conf.cpu().numpy()
        all_conf.extend(confs)
        num_boxes.append(len(confs))

print("Validation tiles:", len(results))
print("Tiles with detections:", len(detected))

if all_conf:
    print(f"Total detections: {len(all_conf)}")
    print(f"Mean confidence: {np.mean(all_conf):.3f}")
    print(f"Min confidence: {np.min(all_conf):.3f}")
    print(f"Max confidence: {np.max(all_conf):.3f}")
    print(f"Median confidence: {np.median(all_conf):.3f}")

print("\nDetection count distribution:")
for n in sorted(set(num_boxes)):
    print(f"  {n} detection(s): {num_boxes.count(n)} tiles")

In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
!{sys.executable} -m pip install torch ultralytics --break-system-packages
